# Uber Pickup Demand Analysis and Prediction

This notebook studies Uber NYC pickup demand using descriptive statistics and a Random Forest Regression model.

## 1. Problem Statement

Uber pickup demand varies with time. The objective of this project is to analyze historical Uber pickup records and predict pickup demand using temporal features such as hour, day, day of the week, and month.

## 2. Objective

- Understand pickup demand patterns
- Use descriptive statistics and correlation analysis
- Build a simple machine learning model
- Predict aggregated pickup demand using temporal features

## 3. Dataset Description

This project uses the existing Uber NYC 2014 data file in the data folder. The raw dataset contains individual pickup events, and the final model uses aggregated demand values by time features.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from scripts.clean_data import clean_uber_data, prepare_ml_dataset
from scripts.visualization import (
    plot_pickups_by_hour,
    plot_pickups_by_day,
    plot_pickups_by_dayofweek,
    plot_monthly_pickups,
    plot_heatmap_day_hour,
    plot_actual_vs_predicted,
    plot_feature_importance,
)

sns.set(style='darkgrid')

project_root = Path.cwd().resolve()
if project_root.name == 'notebook':
    project_root = project_root.parent
print('Project root:', project_root)


In [ ]:
csv_files = sorted((project_root / 'data').glob('uber-raw-data-*.csv'))
raw_df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print('Raw rows:', len(raw_df))
print('Columns:', list(raw_df.columns))
print(raw_df.head())


## 4. Data Exploration

We inspect the dataset, missing values, duplicates, and time range before building the prediction model.

In [ ]:
df = clean_uber_data(csv_files)
print('Cleaned rows:', len(df))
print('Duplicate rows removed:', len(raw_df) - len(df))
print('Date range:', df['Date/Time'].min(), 'to', df['Date/Time'].max())
print(df.isnull().sum())
print(df.dtypes)


## 5. Data Preprocessing

The raw Uber data is cleaned by removing duplicate pickup records and converting the Date/Time column into useful temporal variables such as Hour, Day, DayOfWeek, and Month.

In [ ]:
df['Hour'] = df['Date/Time'].dt.hour
df['Day'] = df['Date/Time'].dt.day
df['DayOfWeek'] = df['Date/Time'].dt.dayofweek
df['DayOfWeekName'] = df['Date/Time'].dt.day_name()
df['Month'] = df['Date/Time'].dt.month
print(df[['Date/Time', 'Hour', 'Day', 'DayOfWeek', 'Month']].head())


## 6. Exploratory Data Analysis

These plots show pickup behavior by hour, day, weekday, month, and day-hour combination.

In [ ]:
plot_pickups_by_hour(df, show=False)
plot_pickups_by_day(df, show=False)
plot_pickups_by_dayofweek(df, show=False)
plot_monthly_pickups(df, show=False)
plot_heatmap_day_hour(df, show=False)
print('EDA plots created.')


## 7. Statistical Analysis

Descriptive statistics summarize the demand pattern, and correlation analysis helps identify relationships between time features and pickup volume.

In [ ]:
ml_df = prepare_ml_dataset(df)
print(ml_df['Pickup_Count'].describe().to_string())
print('---')
print(ml_df[['Hour', 'Day', 'DayOfWeek', 'Pickup_Count']].corr().round(3).to_string())


## 8. ML Dataset Preparation

The raw trip data is aggregated into pickup demand counts for each time group. This makes the problem a simple regression task with one target value per hour-day combination.

In [ ]:
ml_df = prepare_ml_dataset(df)
print(ml_df.head())
print('ML dataset shape:', ml_df.shape)


## 9. Train-Test Split

We split the aggregated dataset into training and testing sets for prediction.

In [ ]:
X = ml_df[['Hour', 'Day', 'DayOfWeek', 'Month']]
y = ml_df['Pickup_Count']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))


## 10. Random Forest Regression

Random Forest Regression is a suitable model because the target is numerical and the relationship between time features and pickup demand is nonlinear.

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print('Prediction preview:', predictions[:5])


## 11. Model Evaluation

The model is evaluated using MAE, MSE, RMSE, and R?.

In [ ]:
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = mse ** 0.5
r2 = r2_score(y_test, predictions)
print(f'MAE: {mae:.4f}')
print(f'MSE: {mse:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R^2: {r2:.4f}')


## 12. Actual vs Predicted Plot

This plot compares the true pickup counts with the model's predictions.

In [ ]:
plot_actual_vs_predicted(y_test, predictions, show=False)
print('Actual vs predicted plot created.')


## 13. Feature Importance

The Random Forest model also reports the importance of each feature when making predictions.

In [ ]:
importance = dict(zip(X.columns, model.feature_importances_))
print(importance)
plot_feature_importance(list(X.columns), model.feature_importances_, show=False)


## 14. Ethical Considerations

- Privacy: transportation data should be handled responsibly.
- Bias: historical records may reflect specific regions and time periods.
- Fairness: the model may not represent every location equally.
- Transparency: predictions are estimates, not ground truth.
- Responsible use: the model supports analysis and planning rather than replacing human judgment.

## 15. Limitations

- The dataset covers only a short historical period.
- Weather and traffic information are not included.
- Holidays and events are not modeled directly.
- The model uses only temporal features.
- Historical patterns may not match future demand.

## 16. Conclusion

The project successfully analyzes Uber pickup demand and builds a Random Forest Regression model for prediction. The model shows strong predictive performance, and it is useful for understanding which time features matter most for demand.

## 17. Future Scope

Future work can improve the model by adding weather data, traffic conditions, holidays, events, and additional location-based features.